In [ ]:
import os

# 1. Check exactly where your notebook is running from
print("Current folder:", os.getcwd())

# 2. List all files in your target data directory to see what is actually there
print("Files in ../data/:", os.listdir("../data/"))

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/ais-2025-01-01")

In [ ]:
df.head()
df.shape
df.columns
df.info()

In [ ]:
df.isna().sum()

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())

print(
    "Duplicate MMSI + timestamp:",
    df.duplicated(
        subset=["mmsi", "base_date_time"]
    ).sum()
)

In [ ]:
invalid_coords = (
    ~df["latitude"].between(-90, 90) |
    ~df["longitude"].between(-180, 180)
)

print("Invalid coordinates:", invalid_coords.sum())

In [ ]:
print("Latitude:")
print(df["latitude"].describe())

print("\nLongitude:")
print(df["longitude"].describe())

In [ ]:
# 1. Remove exact duplicate records
df = df.drop_duplicates()

# 2. Convert timestamp
df["base_date_time"] = pd.to_datetime(
    df["base_date_time"],
    utc=True,
    errors="coerce"
)

# 3. Remove records where timestamp conversion failed
df = df.dropna(subset=["base_date_time"])

# 4. Ensure coordinates are valid
df = df[
    df["latitude"].between(-90, 90) &
    df["longitude"].between(-180, 180)
]

# 5. Sort by vessel and time
df = df.sort_values(
    ["mmsi", "base_date_time"]
)

In [ ]:
dup_mask = df.duplicated(
    subset=["mmsi", "base_date_time"],
    keep=False
)

duplicate_records = df[dup_mask]

print(duplicate_records.shape)
duplicate_records.head(20)

In [ ]:
# After removing exact duplicates
df = df.drop_duplicates()

# Find remaining same-vessel/same-time records
dup_mask = df.duplicated(
    subset=["mmsi", "base_date_time"],
    keep=False
)

same_time = df[dup_mask].copy()

print("Rows involved:", len(same_time))
print("MMSI groups:", same_time.groupby(
    ["mmsi", "base_date_time"]
).ngroups)

In [ ]:
same_time[
    ["mmsi", "base_date_time",
     "longitude", "latitude",
     "sog", "cog", "heading"]
].sort_values(
    ["mmsi", "base_date_time"]
).head(30)

In [ ]:
# 1. Remove exact duplicate rows
df = df.drop_duplicates().copy()

# 2. Convert timestamp to UTC
df["base_date_time"] = pd.to_datetime(
    df["base_date_time"],
    utc=True,
    errors="coerce"
)

# 3. Remove records with invalid timestamps
df = df.dropna(subset=["base_date_time"])

# 4. Validate geographic coordinates
df = df[
    df["latitude"].between(-90, 90) &
    df["longitude"].between(-180, 180)
].copy()

# 5. Flag repeated MMSI + timestamp records
df["same_mmsi_timestamp"] = df.duplicated(
    subset=["mmsi", "base_date_time"],
    keep=False
)

# 6. Sort chronologically for trajectory reconstruction
df = df.sort_values(
    ["mmsi", "base_date_time"]
).reset_index(drop=True)

In [ ]:
print("SOG")
print(df["sog"].describe())

print("\nCOG")
print(df["cog"].describe())

In [ ]:
print("Negative SOG:", (df["sog"] < 0).sum())
print("SOG > 100:", (df["sog"] > 100).sum())

print("Negative COG:", (df["cog"] < 0).sum())
print("COG > 360:", (df["cog"] > 360).sum())

In [ ]:
print("Invalid timestamps:", df["base_date_time"].isna().sum())

print("Earliest:", df["base_date_time"].min())
print("Latest:", df["base_date_time"].max())

print("Timezone:", df["base_date_time"].dt.tz)

In [ ]:
print("MMSI dtype:", df["mmsi"].dtype)

print("MMSI min:", df["mmsi"].min())
print("MMSI max:", df["mmsi"].max())

invalid_mmsi = ~df["mmsi"].astype(str).str.fullmatch(r"\d{9}")

print("Invalid MMSI:", invalid_mmsi.sum())


In [ ]:
df = df.sort_values(
    ["mmsi", "base_date_time"]
).reset_index(drop=True)
time_diff = df.groupby("mmsi")["base_date_time"].diff()

print("Negative time differences:", (time_diff < pd.Timedelta(0)).sum())

In [ ]:
print(time_diff.describe())
print("Gaps > 10 min:", (time_diff > pd.Timedelta(minutes=10)).sum())
print("Gaps > 30 min:", (time_diff > pd.Timedelta(minutes=30)).sum())
print("Gaps > 1 hour:", (time_diff > pd.Timedelta(hours=1)).sum())

In [ ]:
print(
    df["same_mmsi_timestamp"].value_counts()
)

In [ ]:
print(
    df[df["same_mmsi_timestamp"]]
    [["mmsi", "base_date_time", "longitude", "latitude", "sog", "cog"]]
    .head(20)
)


In [ ]:
missing = df.isna().sum()

missing_percentage = (
    df.isna().mean() * 100
).round(2)

quality_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": missing_percentage
})

print(quality_report)

In [ ]:
# Calculate time difference between consecutive observations
# for each vessel
time_diff = df.groupby("mmsi")["base_date_time"].diff()

print("Negative time differences:",
      (time_diff < pd.Timedelta(0)).sum())

print("Zero time differences:",
      (time_diff == pd.Timedelta(0)).sum())

print("Positive time differences:",
      (time_diff > pd.Timedelta(0)).sum())

In [ ]:
zero_time = time_diff == pd.Timedelta(0)

print(
    df.loc[
        zero_time,
        [
            "mmsi",
            "base_date_time",
            "longitude",
            "latitude",
            "sog",
            "cog",
            "heading"
        ]
    ].head(20)
)

In [ ]:
df["prev_lat"] = df.groupby("mmsi")["latitude"].shift(1)
df["prev_lon"] = df.groupby("mmsi")["longitude"].shift(1)
df["prev_time"] = df.groupby("mmsi")["base_date_time"].shift(1)

df["time_diff_seconds"] = (
    df["base_date_time"] - df["prev_time"]
).dt.total_seconds()
print(df["time_diff_seconds"].describe())

In [ ]:
df["ais_gap"] = df["time_diff_seconds"] > 600
print("AIS gaps > 10 minutes:", df["ais_gap"].sum())

In [ ]:
print([
    col for col in [
        "prev_lat",
        "prev_lon",
        "prev_time",
        "time_diff_seconds"
    ]
    if col in df.columns
])

In [ ]:
valid_move = (
    df["prev_lat"].notna() &
    df["prev_lon"].notna() &
    df["time_diff_seconds"].notna() &
    (df["time_diff_seconds"] > 0)
)

print("Valid movements:", valid_move.sum())
print("Invalid movements:", (~valid_move).sum())

In [ ]:
from pyproj import Geod
import numpy as np

geod = Geod(ellps="WGS84")

In [ ]:
df["distance_m"] = np.nan

azimuth1, azimuth2, distance = geod.inv(
    df.loc[valid_move, "prev_lon"].to_numpy(),
    df.loc[valid_move, "prev_lat"].to_numpy(),
    df.loc[valid_move, "longitude"].to_numpy(),
    df.loc[valid_move, "latitude"].to_numpy()
)

df.loc[valid_move, "distance_m"] = distance

In [ ]:
df["implied_speed_knots"] = np.nan

df.loc[valid_move, "implied_speed_knots"] = (
    df.loc[valid_move, "distance_m"]
    / df.loc[valid_move, "time_diff_seconds"]
    * 1.94384
)

In [ ]:
print("implied_speed_knots" in df.columns)

In [ ]:
print(df["implied_speed_knots"].describe())
print(
    "Implied speed > 50 knots:",
    (df["implied_speed_knots"] > 50).sum()
)

print(
    "Implied speed > 100 knots:",
    (df["implied_speed_knots"] > 100).sum()
)

print(
    "Implied speed > 200 knots:",
    (df["implied_speed_knots"] > 200).sum()
)

In [ ]:
print(
    df.nlargest(
        20,
        "implied_speed_knots"
    )[
        [
            "mmsi",
            "base_date_time",
            "prev_time",
            "prev_lat",
            "prev_lon",
            "latitude",
            "longitude",
            "time_diff_seconds",
            "distance_m",
            "implied_speed_knots",
            "sog"
        ]
    ].to_string(index=False)
)

In [ ]:
df["movement_anomaly"] = (
    df["implied_speed_knots"] > 50
)
df["zero_time_gap"] = (
    df["time_diff_seconds"] == 0
)
df["ais_gap"] = (
    df["time_diff_seconds"] > 600
)
print("Movement anomalies:",
      df["movement_anomaly"].sum())

print("Zero-time records:",
      df["zero_time_gap"].sum())

print("AIS gaps >10 min:",
      df["ais_gap"].sum())

In [ ]:
print(
    df.loc[
        df["movement_anomaly"],
        [
            "mmsi",
            "base_date_time",
            "prev_time",
            "time_diff_seconds",
            "distance_m",
            "implied_speed_knots",
            "sog"
        ]
    ].head(20).to_string(index=False)
)

In [ ]:
speed_comparison = df[
    df["implied_speed_knots"].notna() &
    df["sog"].notna()
].copy()

speed_comparison["speed_difference"] = (
    speed_comparison["implied_speed_knots"]
    - speed_comparison["sog"]
)

print(
    speed_comparison[
        [
            "sog",
            "implied_speed_knots",
            "speed_difference"
        ]
    ].describe()
)

In [ ]:
df["valid_movement"] = (
    (df["time_diff_seconds"] > 0) &
    (~df["movement_anomaly"])
)
print(df["valid_movement"].value_counts())

In [ ]:
m4_columns = [
    "mmsi",
    "base_date_time",
    "longitude",
    "latitude",
    "sog",
    "cog",
    "heading",
    "vessel_name",
    "imo",
    "call_sign",
    "vessel_type",
    "prev_lat",
    "prev_lon",
    "prev_time",
    "time_diff_seconds",
    "distance_m",
    "implied_speed_knots",
    "movement_anomaly",
    "zero_time_gap",
    "ais_gap",
    "valid_movement"
]

ais_m4 = df[m4_columns].copy()
print(ais_m4.shape)
print(ais_m4["mmsi"].nunique())
ais_m4.to_csv(
    "ais_m4_clean.csv",
    index=False
)